In [71]:
!pip install trafilatura spacy pandas httpx requests
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [72]:
import os
import re
import json
import time
import random
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import httpx
import requests
import pandas as pd
import trafilatura
import spacy

In [73]:
SEED_URLS = [
    "https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/",
    "https://publications.fifa.com/fr/football-stadiums-guidelines/technical-guideline/stadium-guidelines/information-technology/",
    "https://en.wikipedia.org/wiki/FIFA",
    "https://en.wikipedia.org/wiki/Laws_of_the_Game_(association_football)",
    "https://en.wikipedia.org/wiki/FIFA_World_Cup",
    "https://en.wikipedia.org/wiki/International_Football_Association_Board",
    "https://en.wikipedia.org/wiki/Video_assistant_referee",
    "https://en.wikipedia.org/wiki/UEFA",

]

print(f"{len(SEED_URLS)} seed URLs sélectionnées.")
for url in SEED_URLS:
    print("-", url)

8 seed URLs sélectionnées.
- https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/
- https://publications.fifa.com/fr/football-stadiums-guidelines/technical-guideline/stadium-guidelines/information-technology/
- https://en.wikipedia.org/wiki/FIFA
- https://en.wikipedia.org/wiki/Laws_of_the_Game_(association_football)
- https://en.wikipedia.org/wiki/FIFA_World_Cup
- https://en.wikipedia.org/wiki/International_Football_Association_Board
- https://en.wikipedia.org/wiki/Video_assistant_referee
- https://en.wikipedia.org/wiki/UEFA


In [ ]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

KNOWN_OPEN_DOMAINS = {
    "en.wikipedia.org",
    "fr.wikipedia.org",
    "publications.fifa.com",
}

def is_allowed(url: str, user_agent: str = "*") -> bool:
    parsed = urlparse(url)
    domain = parsed.netloc

    # Domaines publics vérifiés manuellement
    if domain in KNOWN_OPEN_DOMAINS:
        return True

    robots_url = f"{parsed.scheme}://{domain}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except:
        return True

SEED_URLS_FILTERED = [url for url in SEED_URLS if is_allowed(url)]
print(f"{len(SEED_URLS_FILTERED)}/{len(SEED_URLS)} URLs autorisées par robots.txt")

8/8 URLs autorisées par robots.txt


In [75]:
import re

def count_words(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\b\w+\b", text))

def is_useful_page(text: str, min_words: int = 500) -> bool:
    return count_words(text) > min_words

In [76]:
import trafilatura

def fetch_and_extract(url: str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        return None

    text = trafilatura.extract(downloaded)
    if text is None:
        return None

    return {
        "url": url,
        "text": text.strip(),
        "word_count": count_words(text),
        "is_useful": is_useful_page(text)
    }

In [77]:
crawled_pages = []

for url in SEED_URLS_FILTERED:
    result = fetch_and_extract(url)
    if result and result["is_useful"]:
        crawled_pages.append(result)

print(f"{len(crawled_pages)} pages utiles conservées")

8 pages utiles conservées


In [78]:
import json

def save_jsonl(records, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [79]:
save_jsonl(crawled_pages, "crawler_output.jsonl")
print("Fichier sauvegardé : crawler_output.jsonl")

Fichier sauvegardé : crawler_output.jsonl


#5 Phase 2

In [80]:
import spacy

nlp = spacy.load("en_core_web_trf")

In [81]:
TARGET_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

In [82]:
extracted_entities = []

for page in crawled_pages:
    url = page["url"]
    text = page["text"]

    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ in TARGET_LABELS:
            extracted_entities.append({
                "entity": ent.text.strip(),
                "label": ent.label_,
                "source_url": url
            })

print(f"{len(extracted_entities)} entités extraites")

7275 entités extraites


In [83]:
"""
filtered_entities = []
seen = set()

for row in extracted_entities:
    entity = row["entity"].strip()

    if entity == "":
        continue

    key = (entity, row["label"], row["source_url"])
    if key not in seen:
        seen.add(key)
        filtered_entities.append(row)

print(f"{len(filtered_entities)} entités après filtrage")
"""

'\nfiltered_entities = []\nseen = set()\n\nfor row in extracted_entities:\n    entity = row["entity"].strip()\n\n    if entity == "":\n        continue\n\n    key = (entity, row["label"], row["source_url"])\n    if key not in seen:\n        seen.add(key)\n        filtered_entities.append(row)\n\nprint(f"{len(filtered_entities)} entités après filtrage")\n'

In [84]:
import re

filtered_entities = []
seen = set()

def is_clean_entity(text: str) -> bool:
    text = text.strip()
    if not text:
        return False
    if re.match(r'^[•\-\*\d]', text):
        return False
    if len(text) < 2 or len(text) > 100:
        return False
    if text.startswith('\n'):
        return False
    return True

for row in extracted_entities:
    entity = row["entity"].strip()
    if not is_clean_entity(entity):
        continue
    key = (entity, row["label"], row["source_url"])
    if key not in seen:
        seen.add(key)
        filtered_entities.append(row)

print(f"{len(filtered_entities)} entités après filtrage")

2188 entités après filtrage


In [85]:
filtered_entities[:20]

[{'entity': 'FIFA',
  'label': 'ORG',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'Gianni Infantino',
  'label': 'PERSON',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'II',
  'label': 'PERSON',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'eleven years',
  'label': 'DATE',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'WORKING GROUP AND CONTRIBUTORS',
  'label': 'ORG',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'FIFA WORKING GROUP',
  'label': 'ORG',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/'},
 {'entity': 'Colin Smith',
  'label': 'PERSON',
  'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/intr

In [86]:
import pandas as pd

entities_df = pd.DataFrame(filtered_entities)
entities_df.to_csv("extracted_knowledge.csv", index=False, encoding="utf-8")

print("Fichier sauvegardé : extracted_knowledge.csv")

Fichier sauvegardé : extracted_knowledge.csv


In [87]:
sentence_entity_pairs = []

for page in crawled_pages:
    url = page["url"]
    text = page["text"]
    doc = nlp(text)

    for sent in doc.sents:
        sent_ents = [ent for ent in sent.ents if ent.label_ in TARGET_LABELS]

        if len(sent_ents) >= 2:
            sentence_entity_pairs.append({
                "source_url": url,
                "sentence": sent.text.strip(),
                "entities": [(ent.text, ent.label_) for ent in sent_ents]
            })

print(f"{len(sentence_entity_pairs)} phrases avec au moins 2 entités")

977 phrases avec au moins 2 entités


In [ ]:
candidate_relations = []

for page in crawled_pages:
    url = page["url"]
    text = page["text"]
    doc = nlp(text)

    for sent in doc.sents:
        sent_ents = [ent for ent in sent.ents if ent.label_ in TARGET_LABELS]

        if len(sent_ents) < 2:
            continue

        for token in sent:
            if token.pos_ == "VERB":
                has_subject = any(child.dep_ == "nsubj" for child in token.children)
                has_object = any(child.dep_ in {"dobj", "pobj", "attr"} for child in token.children)

                if has_subject and has_object:
                    candidate_relations.append({
                        "source_url": url,
                        "sentence": sent.text.strip(),
                        "verb": token.text,
                        "entities": [(ent.text, ent.label_) for ent in sent_ents]
                    })

print(f"{len(candidate_relations)} relations candidates trouvées")

In [89]:
candidate_relations[:10]

[{'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/',
  'sentence': 'It is eleven years since FIFA last produced some generic stadium guidance in the form of the 5th edition of Football Stadiums: Technical Recommendations and Requirements.',
  'verb': 'produced',
  'entities': [('eleven years', 'DATE'), ('FIFA', 'ORG')]},
 {'source_url': 'https://publications.fifa.com/fr/football-stadiums-guidelines/introduction/',
  'sentence': 'FIFA would like to extend its gratitude to the following people and organisations who contributed to these guidelines:\nFIFA WORKING GROUP\nColin Smith\nHeimo Schirgi\nChristian Stiegler\nKaj Heyral\nGuy Smith\nKate Filochowski\nAlan Ferguson\nCONTRIBUTORS\nARUP\nHFP\nPopulous\nThe Stadium Consultancy\niTurf\nGRAPHIC DESIGN AND PRODUCTION\nJaaf Design\nPHOTO CREDITS\nARUP\nFIFA\nChris Gascoigne\nGetty Images\nGuy Smith\nHufton + Crow\nJames Ewing\nLevel Playing Field\nMilton Keynes Dons F.C.\nMorley von Sternberg\nMurra